# Notebook 01 â€” Stereo Depth (RAFT-Stereo)

Downloads SceneFlow, trains RAFT-Stereo on indoor data, exports ONNX for Jetson TensorRT.

## Cell 01-01 | Mount Drive & Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os, torch

BASE   = '/content/drive/MyDrive/ARGUS'
DS     = f'{BASE}/datasets'
MODELS = f'{BASE}/models'
CKPTS  = f'{BASE}/checkpoints'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE} | Drive: {os.path.exists(BASE)}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda | Drive: True


## Cell 01-02 | Clone RAFT-Stereo Repository

In [4]:
RAFT_DIR = f'{BASE}/raft_stereo'
import subprocess, os

# A partial clone has no HEAD — detect and redo it
def _clone_ok(d): return os.path.exists(d) and os.path.exists(f'{d}/.git/HEAD')

if _clone_ok(RAFT_DIR):
    print('RAFT-Stereo already cloned')
else:
    import shutil
    if os.path.exists(RAFT_DIR):
        print('Incomplete clone found — removing and re-cloning')
        shutil.rmtree(RAFT_DIR)
    print('Cloning RAFT-Stereo (shallow) ...')
    subprocess.run(['git', 'clone', '--depth=1',
                    'https://github.com/princeton-vl/RAFT-Stereo', RAFT_DIR], check=True)
    print('RAFT-Stereo cloned')

os.chdir(RAFT_DIR)
subprocess.run(['pip', 'install', '-q', 'einops'], check=True)
print('RAFT-Stereo directory:'); os.system('ls')

RAFT-Stereo already exists
RAFT-Stereo directory:


0

## Cell 01-03 | Download SceneFlow Dataset

In [5]:
# Download RAFT-Stereo pretrained weights via the repo's own script (gdown).
# gdown --continue resumes interrupted downloads automatically.
import subprocess, os

WEIGHTS_DIR = f'{MODELS}/depth'
os.makedirs(WEIGHTS_DIR, exist_ok=True)
CKPT_PATH = f'{WEIGHTS_DIR}/raftstereo-realtime.pth'
MIN_MB    = 20  # checkpoint is ~25 MB

def _ckpt_ok(p, min_mb):
    return os.path.exists(p) and os.path.getsize(p)/1e6 >= min_mb

if _ckpt_ok(CKPT_PATH, MIN_MB):
    print(f'✓ Checkpoint ready ({os.path.getsize(CKPT_PATH)/1e6:.1f} MB)')
else:
    if os.path.exists(CKPT_PATH):
        print('Partial checkpoint detected — removing'); os.remove(CKPT_PATH)
    subprocess.run(['pip', 'install', '-q', 'gdown'], check=True)
    os.chdir(RAFT_DIR)
    ret = subprocess.run(['bash', 'download_models.sh'],
                         capture_output=True, text=True)
    print(ret.stdout[-2000:] or '(no stdout)')
    if ret.returncode != 0:
        print('stderr:', ret.stderr[-500:])
    # Move .pth files from repo/models/ → MODELS/depth/
    repo_models = f'{RAFT_DIR}/models'
    if os.path.isdir(repo_models):
        for fn in os.listdir(repo_models):
            if fn.endswith('.pth'):
                os.replace(f'{repo_models}/{fn}', f'{WEIGHTS_DIR}/{fn}')
                print(f'Moved {fn} → {WEIGHTS_DIR}/')

if _ckpt_ok(CKPT_PATH, MIN_MB):
    print(f'✅ raftstereo-realtime.pth ready ({os.path.getsize(CKPT_PATH)/1e6:.1f} MB)')
else:
    print('⚠ Checkpoint still missing — check stderr above')

SceneFlow already downloaded


0

## Cell 01-04 | Download Pretrained RAFT-Stereo Weights

In [ ]:
import subprocess, os, zipfile, tarfile as _tarfile

def _safe_dl(url, dest, min_mb=None):
    """wget -c resume. If file exists but too small, delete and restart."""
    if os.path.exists(dest):
        mb = os.path.getsize(dest) / 1e6
        if min_mb and mb < min_mb * 0.90:
            print(f"  Partial file {os.path.basename(dest)} ({mb:.0f}/{min_mb} MB) — removing")
            os.remove(dest)
        else:
            print(f"  ✓ {os.path.basename(dest)} ({mb:.0f} MB)")
            return True
    print(f"  Downloading {os.path.basename(dest)} ...")
    r = subprocess.run(["wget", "-c", "--show-progress", "-O", dest, url])
    return r.returncode == 0

def _verify(path):
    """Return True if zip/tar is intact, delete it and return False if corrupt."""
    try:
        if path.endswith(".zip"):
            with zipfile.ZipFile(path) as z:
                bad = z.testzip()
                if bad: raise ValueError(bad)
        elif path.endswith((".tar.gz", ".tgz", ".tar")):
            with _tarfile.open(path) as t:
                t.getmembers()
        return True
    except Exception as e:
        print(f"  Corrupt archive ({e}) — removing for clean re-download")
        os.remove(path)
        return False

def _extract(path, dest):
    print(f"  Extracting {os.path.basename(path)} ...")
    if path.endswith(".zip"):
        with zipfile.ZipFile(path) as z: z.extractall(dest)
    else:
        subprocess.run(["tar", "-xf", path, "-C", dest], check=True)


# Prepare custom AR0234 stereo data directory.
# Optionally pull ETH3D indoor SLAM subset (~500 MB) as extra training data.

CUSTOM_STEREO = f'{DS}/custom_stereo'
for split in ('train', 'val'):
    for sub in ('left', 'right', 'disp'):
        os.makedirs(f'{CUSTOM_STEREO}/{split}/{sub}', exist_ok=True)
print('Custom stereo directory structure ready.')

USE_ETH3D = False  # flip to True to download ~500 MB ETH3D indoor supplement

if USE_ETH3D:
    ETH_DIR = f'{DS}/eth3d_indoor'
    ETH_ZIP = f'{ETH_DIR}/eth3d_slam.zip'
    ETH_URL = 'https://www.eth3d.net/data/slam/training_area_sets/slam_training_areas.zip'
    ETH_FLAG= f'{ETH_DIR}/eth3d_done.flag'
    os.makedirs(ETH_DIR, exist_ok=True)
    if os.path.exists(ETH_FLAG):
        print('✓ ETH3D already extracted')
    else:
        ok = _safe_dl(ETH_URL, ETH_ZIP, min_mb=400)
        if ok and _verify(ETH_ZIP):
            _extract(ETH_ZIP, ETH_DIR)
            open(ETH_FLAG, 'w').close()
            print('✅ ETH3D ready')
        elif not ok:
            print('⚠ ETH3D download failed — retry this cell')

left_dir = f'{CUSTOM_STEREO}/train/left'
n = len([f for f in os.listdir(left_dir) if f.endswith(('.png','.jpg'))]) if os.path.exists(left_dir) else 0
print(f'AR0234 training pairs found: {n}')
if n == 0: print('Add AR0234 stereo pairs before running the training cell.')

## Cell 01-05 | Custom Indoor Dataset Loader

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import numpy as np
import albumentations as A
from albumentations.pytorch import ToTensorV2

class StereoDataset(Dataset):
    # Custom stereo dataset for AR0234 indoor image pairs
    def __init__(self, root, split='train', img_size=(480, 640)):
        self.left_dir  = f'{root}/{split}/left'
        self.right_dir = f'{root}/{split}/right'
        self.disp_dir  = f'{root}/{split}/disp'
        self.img_size  = img_size
        self.samples   = sorted([f for f in os.listdir(self.left_dir) if f.endswith(('.png', '.jpg'))])
        print(f'  {split}: {len(self.samples)} stereo pairs found')
        self.transform = A.Compose([
            A.RandomCrop(img_size[0], img_size[1]),
            A.ColorJitter(brightness=0.2, contrast=0.2, p=0.5),
            A.ToFloat(max_value=255),
            ToTensorV2(),
        ], additional_targets={'right': 'image'})

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        name  = self.samples[idx]
        stem  = os.path.splitext(name)[0]
        left  = np.array(Image.open(f'{self.left_dir}/{name}').convert('RGB'))
        right = np.array(Image.open(f'{self.right_dir}/{name}').convert('RGB'))
        disp  = np.load(f'{self.disp_dir}/{stem}.npy').astype(np.float32)
        out   = self.transform(image=left, right=right)
        disp_t = torch.from_numpy(disp[:self.img_size[0], :self.img_size[1]]).unsqueeze(0)
        return {'left': out['image'], 'right': out['right'], 'disparity': disp_t}

CUSTOM_STEREO = f'{DS}/custom_stereo'
for d in [f'{CUSTOM_STEREO}/train/left', f'{CUSTOM_STEREO}/train/right', f'{CUSTOM_STEREO}/train/disp']:
    os.makedirs(d, exist_ok=True)
print('Custom stereo dataset structure ready.')
print('Add AR0234 stereo pairs before running the training cell.')

## Cell 01-06 | RAFT-Stereo Training Loop

In [ ]:
import sys
sys.path.insert(0, RAFT_DIR)
from core.raft_stereo import RAFTStereo
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
import torch.nn.functional as F

args = type('Args', (), {
    'hidden_dims': [128]*3, 'context_dims': [128]*3,
    'corr_implementation': 'reg', 'shared_backbone': False,
    'corr_levels': 4, 'corr_radius': 4, 'n_downsample': 2,
    'slow_fast_gru': True, 'n_gru_layers': 3, 'mixed_precision': True,
})()

model = RAFTStereo(args).to(DEVICE)
ckpt  = torch.load(f'{MODELS}/depth/raftstereo-realtime.pth', map_location=DEVICE)
model.load_state_dict(ckpt, strict=False)
print('Pretrained weights loaded')

CUSTOM_STEREO = f'{DS}/custom_stereo'
has_custom    = len(os.listdir(f'{CUSTOM_STEREO}/train/left')) > 0
dataset       = StereoDataset(CUSTOM_STEREO, 'train', img_size=(320, 480))
loader        = DataLoader(dataset, batch_size=2, shuffle=True, num_workers=4, pin_memory=True, drop_last=True)

optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
scheduler = OneCycleLR(optimizer, max_lr=1e-4, steps_per_epoch=len(loader), epochs=20, pct_start=0.05)
scaler    = torch.cuda.amp.GradScaler()

EPOCHS    = 20
CKPT_PATH = f'{CKPTS}/raft_stereo_indoor'
os.makedirs(CKPT_PATH, exist_ok=True)

model.train()
for epoch in range(EPOCHS):
    epoch_loss = 0.0
    for i, batch in enumerate(loader):
        left = batch['left'].to(DEVICE)
        right = batch['right'].to(DEVICE)
        disp  = batch['disparity'].to(DEVICE)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            preds = model(left, right, iters=12)
            loss  = 0.0
            for j, pred in enumerate(preds):
                w    = 0.9 ** (len(preds) - j - 1)
                mask = (disp > 0) & (disp < 192)
                loss += w * F.smooth_l1_loss(pred[mask], disp[mask])
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update(); scheduler.step()
        epoch_loss += loss.item()
        if i % 10 == 0:
            print(f'Epoch {epoch+1}/{EPOCHS} | Step {i}/{len(loader)} | Loss: {loss.item():.4f}')
    print(f'Epoch {epoch+1} avg loss: {epoch_loss/len(loader):.4f}')
    if (epoch + 1) % 5 == 0:
        torch.save({'epoch': epoch+1, 'model_state': model.state_dict()},
                   f'{CKPT_PATH}/epoch_{epoch+1}.pth')

torch.save(model.state_dict(), f'{MODELS}/depth/raft_stereo_final.pth')
print('Training complete. Final model saved to Drive.')

## Cell 01-07 | Export to ONNX

In [ ]:
model.eval()
H, W         = 480, 640
dummy_left   = torch.randn(1, 3, H, W).to(DEVICE)
dummy_right  = torch.randn(1, 3, H, W).to(DEVICE)
ONNX_PATH    = f'{EXPORTS}/tensorrt/raft_stereo_640x480.onnx'
os.makedirs(os.path.dirname(ONNX_PATH), exist_ok=True)

class RAFTStereoExport(torch.nn.Module):
    def __init__(self, m): super().__init__(); self.model = m
    def forward(self, l, r): return self.model(l, r, iters=12, test_mode=True)

with torch.no_grad():
    torch.onnx.export(
        RAFTStereoExport(model), (dummy_left, dummy_right), ONNX_PATH,
        input_names=['left', 'right'], output_names=['disparity'],
        dynamic_axes={'left': {0: 'batch'}, 'right': {0: 'batch'}, 'disparity': {0: 'batch'}},
        opset_version=17, do_constant_folding=True,
    )
print(f'ONNX exported: {ONNX_PATH}')
print(f'Size: {os.path.getsize(ONNX_PATH)/1e6:.1f} MB')
print('On Jetson: trtexec --onnx=raft_stereo_640x480.onnx --saveEngine=raft_stereo_fp16.trt --fp16')